# 12.11 · Prompt 工程 / Prompt Engineering

> **课程定位 / Where this fits**
> 第 11 课，**Part 12**。用好 LLM 最便宜、最快、最常用的方式——**不改一个权重**。
> Lesson 11, **Part 12**. The cheapest, fastest, most common way to use LLMs — **without touching a single weight**.
>
> 同一个大模型，**问法不同，效果天差地别**。**Prompt 工程**就是研究"怎么写提示词"让 LLM 发挥最好。关键技巧：**zero-shot / few-shot(给几个例子) / 思维链 CoT(让它一步步想) / 自洽(采样多次投票) / ReAct(边想边用工具)**。这是当今最实用的"软技能"——很多场景一个好 prompt 就够，无需微调。本课讲清每种技巧、给出可复制的**prompt 模板**，并用一个**可运行的自洽(self-consistency)实验**揭示"采样多次取多数"为何有效。
> The same LLM, **asked differently, performs wildly differently**. **Prompt engineering** studies how to phrase prompts to get the best from an LLM. Key techniques: **zero-shot / few-shot (give examples) / chain-of-thought CoT (think step by step) / self-consistency (sample many, vote) / ReAct (reason + use tools)**. The most practical "soft skill" today — often a good prompt suffices, no fine-tuning. We explain each, give reusable **prompt templates**, and use a **runnable self-consistency experiment** to reveal why "sample many and take the majority" works.
>
> 💼 **实战/面试视角**："few-shot/CoT/self-consistency/ReAct 原理与适用 / 为什么 CoT 有效 / prompt 注入风险" 是 LLM 应用岗高频。
> 💼 **Practical/interview angle:** "few-shot/CoT/self-consistency/ReAct / why CoT works / prompt injection" — frequent for LLM application roles.

> 📐 **符号约定 / Notation**
> - zero-shot —— 不给例子直接问 / ask with no examples
> - few-shot —— prompt 里放几个示范 / put a few demos in the prompt
> - CoT —— chain-of-thought, 让模型显式写出推理步骤 / chain-of-thought

> 💡 **面试相关 / Interview-relevant**
> - "few-shot / in-context learning 是什么"（出镜率 ★★★★）
> - "思维链 CoT 为什么能提升推理"（★★★★★）
> - "self-consistency 为什么有效"（★★★★）
> - "ReAct / 工具调用"（★★★★）
> - "prompt 注入 / 越狱 风险"（★★★）

---

## 学习目标 / Learning Objectives
1. 掌握 zero-shot / few-shot 及 in-context learning。
   Master zero-shot / few-shot and in-context learning.
2. 理解**思维链(CoT)** 为何提升推理。
   Understand why chain-of-thought improves reasoning.
3. 用实验理解 **self-consistency** 为何有效。
   Use an experiment to understand why self-consistency works.
4. 了解 ReAct / 工具调用与 prompt 风险。
   Know ReAct / tool use and prompt risks.

## 目录 / TOC
1. [zero-shot / few-shot 与 in-context learning ⭐](#1)
2. [思维链 CoT ⭐](#2)
3. [自洽 self-consistency（可运行实验）⭐](#3)
4. [ReAct、风险与小结 ⭐](#4)


<a id="1"></a>
## 1. zero-shot / few-shot 与 in-context learning ⭐ / Zero/Few-Shot & In-Context Learning

(注：本机没有可调用的大模型 API，下面用**真实的 prompt 模板**讲清技巧，并在 §3 用一个可运行实验揭示核心机制。)
(Note: no callable LLM API here; we explain techniques with **real prompt templates** and reveal the core mechanism with a runnable experiment in §3.)

- **zero-shot(零样本)**：不给任何例子，直接下指令。靠模型预训练学到的通用能力。
  **Zero-shot:** no examples, just an instruction. Relies on pretrained general ability.
  ```
  把下面评论分类为 正面/负面：
  "这家餐厅太难吃了。" →
  ```
- **few-shot(少样本)**：在 prompt 里放**几个输入-输出示范**，模型**照着模式做**——这就是著名的 **in-context learning(上下文学习)**：模型**不更新权重**，仅靠 prompt 里的例子"现学现用"。这是大模型规模够大时的涌现能力(12.8)。
  **Few-shot:** include a **few input-output demos**; the model **follows the pattern** — the famous **in-context learning**: no weight updates, it "learns on the fly" from prompt examples. An emergent ability at scale (12.8).
  ```
  评论 → 情感
  "环境很棒，服务周到" → 正面
  "等了一小时还上错菜" → 负面
  "性价比超高，会再来" → 正面
  "太难吃了" →            ← 模型照着前面的模式输出"负面"
  ```

**为什么 few-shot 有用**：例子帮模型**锁定任务格式与标签空间**，消除歧义(它知道你要"正面/负面"而不是写影评)。
**Why few-shot helps:** examples help the model **lock onto the task format and label space**, removing ambiguity.


<a id="2"></a>
## 2. 思维链 CoT ⭐ / Chain-of-Thought

对**需要推理**的问题(数学、逻辑、多步推断)，直接让模型蹦出答案往往出错。**思维链(Chain-of-Thought, CoT)** 让模型**先一步步写出推理过程，再给答案**——准确率大幅提升。
For **reasoning** problems (math, logic, multi-step inference), making the model blurt the answer often fails. **Chain-of-Thought (CoT)** has the model **write out reasoning step by step before answering** — greatly boosting accuracy.

最神奇的是一句**魔法咒语**：在 prompt 末尾加 **"Let's think step by step"(让我们一步一步思考)**，就能触发 zero-shot CoT。
The magic part: appending **"Let's think step by step"** triggers zero-shot CoT.

```
问: 食堂有23个苹果, 用掉20个做午餐, 又买了6个, 现在有几个?

直接回答(易错): "27"        ← 蹦答案, 可能算错

思维链 (Let's think step by step):
  原来有 23 个。
  用掉 20 个 → 23 − 20 = 3 个。
  又买 6 个 → 3 + 6 = 9 个。
  答案: 9                    ← 拆成小步, 每步简单, 不易错
```

**为什么 CoT 有效**(面试核心)：① 把一个难的多步问题**分解成多个简单步骤**，每步模型更可能做对；② 生成中间步骤等于**给模型更多"思考的计算量"**(更多 token = 更多前向计算)，复杂问题需要这些"草稿纸"。CoT 是激发大模型推理能力最重要的 prompt 技巧。
**Why CoT works** (interview core): ① it **decomposes** a hard multi-step problem into simple steps, each more likely correct; ② generating intermediate steps gives the model **more "compute to think"** (more tokens = more forward computation) — hard problems need this "scratch paper." CoT is the most important prompting trick for eliciting reasoning.


<a id="3"></a>
## 3. 自洽 self-consistency（可运行实验）⭐ / Self-Consistency (Runnable Experiment)

**自洽(self-consistency)** 在 CoT 基础上更进一步：对同一问题，**用带随机性的采样生成多条不同的推理链**，得到多个答案，再**取出现次数最多的那个(多数投票)**。
**Self-consistency** builds on CoT: for the same question, **sample multiple different reasoning chains** (with randomness), get multiple answers, and **take the most frequent one (majority vote)**.

**为什么有效**：不同推理路径可能在不同地方出错，但**正确答案往往是多条路径的共同终点**。错误是分散的、正确是集中的——多数投票就能把正确答案"投"出来。这本质就是**集成(ensembling)**(呼应 Part 8)。
**Why it works:** different reasoning paths err in different places, but the **correct answer is often the common endpoint of many paths**. Errors scatter, correct concentrates — majority vote surfaces the correct answer. This is essentially **ensembling** (echoing Part 8).

下面用一个**可运行实验**揭示这个机制：把"一条推理链"建模为"**有 $p$ 概率得到正确答案、否则随机错一个**"的采样器。看**采样 $k$ 条 + 多数投票**的准确率如何随 $k$ 上升。
A **runnable experiment** reveals the mechanism: model "one reasoning chain" as a sampler that's **correct with prob $p$, else a random wrong answer**. Watch how **sample-$k$ + majority vote** accuracy rises with $k$.


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
sns.set_theme(style="whitegrid"); np.random.seed(0)

def one_chain(correct, p, n_options=5):
    """模拟一条推理链: 概率 p 得到正确答案, 否则随机选一个错误答案 / one CoT sample."""
    if np.random.rand() < p: return correct
    wrong = [o for o in range(n_options) if o != correct]
    return np.random.choice(wrong)

def self_consistency_acc(p, k, trials=4000, n_options=5):
    correct_count = 0
    for _ in range(trials):
        true = np.random.randint(n_options)
        answers = [one_chain(true, p) for _ in range(k)]   # 采样 k 条链 / sample k chains
        majority = Counter(answers).most_common(1)[0][0]   # 多数投票 / majority vote
        correct_count += (majority == true)
    return correct_count / trials

ks = [1, 3, 5, 9, 15, 25]
fig, ax = plt.subplots(figsize=(8, 4.5))
for p in [0.4, 0.5, 0.6]:
    accs = [self_consistency_acc(p, k) for k in ks]
    ax.plot(ks, accs, "o-", label=f"单链正确率 p={p}")
    print(f"单链正确率 p={p}: 1条={accs[0]:.2f} → {ks[-1]}条多数投票={accs[-1]:.2f}")
ax.axhline(0.5, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("采样链数 k"); ax.set_ylabel("多数投票后的准确率"); ax.legend()
ax.set_title("self-consistency: 采样多条推理链取多数 → 准确率随 k 上升(当单链>随机时)")
plt.tight_layout(); plt.show()
print("\n只要单条链的正确率 > 随机(且错误分散), 多数投票就能显著拉高准确率 → 这就是 self-consistency")
print("本质=集成(Part 8): 多个'弱推理'投票出强答案; 代价是多倍推理算力")


<a id="4"></a>
## 4. ReAct、风险与小结 ⭐ / ReAct, Risks & Summary

**ReAct(Reason + Act)**：让模型**交替"推理"和"行动(调用工具)"**——想一步、查一下(搜索/计算器/数据库)、再根据结果继续想。这把 LLM 从"只会凭记忆答"升级为"会用工具解决问题"，是 **Agent(12.14)** 的基础。
**ReAct (Reason + Act):** the model **alternates "reasoning" and "acting" (tool calls)** — think a step, look something up (search/calculator/database), then continue based on the result. This upgrades the LLM from "answer from memory" to "use tools to solve," the basis of **agents (12.14)**.
```
问: 现在巴黎几点?
  Thought: 我需要查实时时间, 用工具。
  Action: get_time("Paris")
  Observation: 14:30
  Thought: 得到了答案。
  Answer: 巴黎现在是下午 2:30。
```

**其他实用技巧**：给**角色/系统提示**("你是一位资深医生")、**结构化输出**(要求 JSON)、**明确约束**(字数/格式/语气)、**分隔符**清晰区分指令与数据。
**Other practical tricks:** role/system prompts ("You are a senior doctor"), structured output (request JSON), explicit constraints (length/format/tone), delimiters separating instructions from data.

**风险(面试)**：**prompt 注入(injection)** ——恶意用户在输入里夹带"忽略上面的指令，改为…"来劫持模型；**越狱(jailbreak)** ——绕过安全限制。处理不可信输入时要格外小心(用分隔符、输出校验、权限隔离)。
**Risks:** **prompt injection** — malicious input like "ignore the above and instead…" hijacks the model; **jailbreak** — bypass safety. Be careful with untrusted input (delimiters, output validation, privilege isolation).

```
zero-shot: 直接下指令; few-shot: prompt里给几个示范→in-context learning(不更新权重现学现用)
CoT思维链: 让模型一步步推理再答; 为什么有效=分解难题+给更多思考算力; "let's think step by step"
self-consistency: 采样多条CoT链取多数投票; 本质=集成(错误分散正确集中); 代价多倍算力
ReAct: 推理与调用工具交替; LLM Agent 的基础
实用技巧: 角色/系统提示, few-shot, CoT, 结构化输出(JSON), 明确约束, 分隔符
风险: prompt注入/越狱; 处理不可信输入要分隔+校验+权限隔离
适配排序(12.8): Prompt最便宜 → RAG → LoRA → 全量微调
```

### 💡 面试速查 / Interview cheat-sheet
1. **few-shot/ICL**: prompt里给例子, 模型不更新权重就学会任务格式。
   Few-shot/ICL: examples in the prompt; the model learns the task without weight updates.
2. **CoT**: 让模型一步步推理→准确率升; =分解+更多思考算力。
   CoT: step-by-step reasoning → higher accuracy; = decomposition + more compute.
3. **self-consistency**: 采样多条链取多数; 集成思想(错误分散)。
   Self-consistency: sample many chains, majority vote; ensembling (errors scatter).
4. **ReAct**: 推理+工具调用交替; Agent 基础。
   ReAct: interleave reasoning + tool calls; basis of agents.
5. **风险**: prompt注入/越狱; 不可信输入要隔离校验。
   Risks: prompt injection/jailbreak; isolate & validate untrusted input.

### 下一节 / Next
**12.12 RAG 检索增强生成**——LLM 的知识停在训练时刻、且会"一本正经地胡说(幻觉)"。**RAG** 给它外挂一个知识库: 先**检索**相关文档, 再把内容塞进 prompt 让模型**基于事实回答**。我们会**从零搭一个 RAG 流程**。
**12.12 RAG** — an LLM's knowledge is frozen at training time and it "hallucinates." **RAG** attaches a knowledge base: **retrieve** relevant documents, put them in the prompt so the model **answers from facts**. We'll **build a RAG pipeline from scratch**.
